In [ ]:
"""
=============================================================================
Part A: Data Preprocessing and Tokenizer Training
CSE556 NLP, Winter 2026 — Assignment 2
=============================================================================
This script:
1. Splits the corpus into train/val
2. Trains a BPE tokenizer on the training split
3. Tokenizes both splits and saves them
4. Prepares classification data
=============================================================================
"""

import os
import glob
import random
import pickle
import numpy as np
import pandas as pd
import sentencepiece as spm



class ConfigA:

    corpus_dir          = r"C:\Users\gaurm\OneDrive\Desktop\hindi_corpus\hindi_corpus\train"
    classification_file = r"C:\Users\gaurm\OneDrive\Desktop\train.csv"
    
    tokenizer_prefix    = "hindi_bpe_trainsplit"
    
    
    corpus_train_frac = 0.80
    cls_train_frac    = 0.80
    
   
    vocab_size = 5000
    

    block_size = 128
    
  
    seed = 42
    
   
    train_tokens_file = "train_tokens.pkl"
    val_tokens_file   = "val_tokens.pkl"
    cls_data_file     = "classification_data.pkl"


cfg = ConfigA()


random.seed(cfg.seed)
np.random.seed(cfg.seed)




def list_corpus_files(corpus_dir: str):
   
    files = sorted(glob.glob(os.path.join(corpus_dir, "*")))
    if len(files) == 0:
        raise FileNotFoundError(f"No files found in corpus_dir: {corpus_dir}")
    return files


def read_corpus_files(file_paths):
   
    parts = []
    total_chars = 0
    for fpath in file_paths:
        with open(fpath, "r", encoding="utf-8") as f:
            txt = f.read()
            parts.append(txt)
            total_chars += len(txt) + 1
    text = "\n".join(parts)
    return text, total_chars


def split_corpus_files(corpus_dir: str, train_frac: float):
  
    files = list_corpus_files(corpus_dir)
    split_idx = int(train_frac * len(files))
    train_files = files[:split_idx]
    val_files   = files[split_idx:]
    
    train_text, train_chars = read_corpus_files(train_files)
    val_text,   val_chars   = read_corpus_files(val_files)
    
    print(f"[Corpus]  Loaded {len(files)} files total")
    print(f"[Corpus]  Train files: {len(train_files)}  |  Val files: {len(val_files)}")
    print(f"[Corpus]  Train chars: {train_chars:,}  |  Val chars: {val_chars:,}")
    return train_text, val_text



def train_tokenizer(text: str, prefix: str, vocab_size: int):
  
    tmp = f"{prefix}_raw.txt"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    
    spm.SentencePieceTrainer.Train(
        input=tmp,
        model_prefix=prefix,
        vocab_size=vocab_size,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=3,
    )
    print(f"[Tokenizer]  Trained BPE on training split  |  vocab_size = {vocab_size}")


def load_tokenizer(prefix: str) -> spm.SentencePieceProcessor:
   
    sp = spm.SentencePieceProcessor()
    sp.Load(f"{prefix}.model")
    return sp


def encode(sp, text: str):
    """Encode text to token IDs."""
    return sp.EncodeAsIds(text)


def decode(sp, ids):
    """Decode token IDs to text."""
    return sp.DecodeIds(ids)




def load_classification_data(filepath: str):
    """
    Read classification CSV and return (texts, labels).
    
    Target mapping required by assignment:
        0 = Negative, 1 = Neutral, 2 = Positive
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Classification file not found: {filepath}")
    
    df = pd.read_csv(filepath)
    print(f"[Classification]  Columns: {list(df.columns)}")
    
   
    text_col, label_col = None, None
    for c in df.columns:
        cl = c.strip().lower()
        if cl in ("review", "text", "sentence", "hindi", "review_body"):
            text_col = c
        if cl in ("label", "sentiment", "class", "rating", "star", "experience"):
            label_col = c
    
    if text_col is None:
        text_col = df.columns[0]
    if label_col is None:
        label_col = df.columns[-1]
    
    print(f"[Classification]  Using  text = '{text_col}',  label = '{label_col}'")
    
    texts = df[text_col].astype(str).tolist()
    raw_labels = df[label_col].tolist()
    
    unique = sorted(set(raw_labels))
    label_map = {}
    
   
    normalized_unique = {str(u).strip() for u in unique}
    if normalized_unique == {"0", "1", "2"}:
        label_map = {u: int(str(u).strip()) for u in unique}
    else:
        # Case 2: textual labels
        for u in unique:
            s = str(u).strip().lower()
            if "neg" in s:
                label_map[u] = 0
            elif "neu" in s:
                label_map[u] = 1
            elif "pos" in s:
                label_map[u] = 2
        
      
        if len(label_map) != len(unique):
            label_map = {u: i for i, u in enumerate(unique)}
    
    labels = [label_map[l] for l in raw_labels]
    print(f"[Classification]  {len(texts)} samples  |  label_map = {label_map}")
    return texts, labels




def main():
    print("=" * 70)
    print(" PART A: Data Preprocessing and Tokenization")
    print("=" * 70)
    
  
    print("\n[Step 1] Splitting corpus...")
    train_corpus_text, val_corpus_text = split_corpus_files(
        cfg.corpus_dir,
        cfg.corpus_train_frac
    )
    
    
    print("\n[Step 2] Training/Loading tokenizer...")
    if not os.path.exists(f"{cfg.tokenizer_prefix}.model"):
        train_tokenizer(train_corpus_text, cfg.tokenizer_prefix, cfg.vocab_size)
    else:
        print(f"[Tokenizer]  Model already exists, skipping training")
    
    sp = load_tokenizer(cfg.tokenizer_prefix)
    print(f"[Tokenizer]  Vocab size = {sp.GetPieceSize()}")
    
  
    sample = "भारत एक महान देश है"
    sample_ids = encode(sp, sample)
    print(f"[Tokenizer]  '{sample}'  ->  {sample_ids}")
    print(f"[Tokenizer]  decode back ->  '{decode(sp, sample_ids)}'")
    
  
    print("\n[Step 3] Tokenizing corpus...")
    train_ids = encode(sp, train_corpus_text)
    val_ids   = encode(sp, val_corpus_text)
    
    print(f"[Corpus]  Train tokens: {len(train_ids):,}")
    print(f"[Corpus]  Val tokens  : {len(val_ids):,}")
    
   
    with open(cfg.train_tokens_file, "wb") as f:
        pickle.dump(train_ids, f)
    print(f"[Corpus]  Saved -> {cfg.train_tokens_file}")
    
    with open(cfg.val_tokens_file, "wb") as f:
        pickle.dump(val_ids, f)
    print(f"[Corpus]  Saved -> {cfg.val_tokens_file}")
    
   
    print("\n[Step 4] Processing classification data...")
    cls_texts, cls_labels = load_classification_data(cfg.classification_file)
    

    idxs = list(range(len(cls_texts)))
    random.shuffle(idxs)
    split_idx = int(cfg.cls_train_frac * len(idxs))
    
    train_cls_data = {
        'texts':  [cls_texts[i]  for i in idxs[:split_idx]],
        'labels': [cls_labels[i] for i in idxs[:split_idx]]
    }
    
    val_cls_data = {
        'texts':  [cls_texts[i]  for i in idxs[split_idx:]],
        'labels': [cls_labels[i] for i in idxs[split_idx:]]
    }
    
    cls_data = {
        'train': train_cls_data,
        'val': val_cls_data,
        'pad_id': sp.pad_id()
    }
    

    with open(cfg.cls_data_file, "wb") as f:
        pickle.dump(cls_data, f)
    print(f"[Classification]  Saved -> {cfg.cls_data_file}")
    print(f"[Classification]  Train: {len(train_cls_data['texts'])}  |  Val: {len(val_cls_data['texts'])}")
    
    # Summary
    print("\n" + "=" * 70)
    print(" PREPROCESSING COMPLETE")
    print("=" * 70)
    print("  Generated files:")
    print(f"    • {cfg.tokenizer_prefix}.model")
    print(f"    • {cfg.tokenizer_prefix}.vocab")
    print(f"    • {cfg.train_tokens_file}")
    print(f"    • {cfg.val_tokens_file}")
    print(f"    • {cfg.cls_data_file}")
    print("\n  Use these files in Part B (Language Model Training)")


if __name__ == "__main__":
    main()

In [ ]:
"""
=============================================================================
Part B: GPT-2 Language Model Training
CSE556 NLP, Winter 2026 — Assignment 2
=============================================================================
This script:
1. Loads preprocessed token data from Part A
2. Creates language modeling datasets
3. Trains the decoder-only transformer (GPT-style)
4. Saves model weights for use in Part C
=============================================================================
"""

import os
import math
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import sentencepiece as spm



class ConfigB:
    """Configuration for language model training."""
    
    # ----- Paths -----
    tokenizer_prefix    = "hindi_bpe_trainsplit"
    train_tokens_file   = "train_tokens.pkl"
    val_tokens_file     = "val_tokens.pkl"
    
    # ----- Model architecture -----
    block_size = 128
    d_model    = 256
    n_heads    = 4
    n_layers   = 10
    dropout    = 0.1
    vocab_size = 5000  
    
    # ----- Training -----
    lm_batch_size = 64
    lm_epochs     = 3
    lm_lr         = 3e-4
    lm_stride     = 64  
    
    # ----- General -----
    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Dataloader settings
    num_workers = 0 if torch.cuda.is_available() else 0
    pin_memory  = False if torch.cuda.is_available() else False
    
    # ----- Output -----
    model_save_path = "lm_model.pt"
    loss_plot_path  = "lm_loss.png"
    generated_text_path = "generated_text.txt"


cfg = ConfigB()


torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = True




class LMDataset(Dataset):
    """
    Produces (x, y) pairs for next-token prediction.
    y is x shifted by one position.
    
    stride=1 gives all overlapping windows.
    stride>1 is faster but uses fewer windows.
    """
    def __init__(self, data: torch.Tensor, block_size: int, stride: int = 1):
        self.data       = data
        self.block_size = block_size
        self.stride     = stride
        
        if len(self.data) <= self.block_size:
            raise ValueError("Data is too short for the chosen block_size.")
        
        self.n_samples = ((len(self.data) - self.block_size - 1) // self.stride) + 1
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        start = idx * self.stride
        x = self.data[start : start + self.block_size]
        y = self.data[start + 1 : start + self.block_size + 1]
        return x, y




class CausalSelfAttention(nn.Module):
    """
    Multi-head self-attention with a causal mask.
    Uses F.scaled_dot_product_attention with is_causal=True.
    """
    def __init__(self, d_model: int, n_heads: int, block_size: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        
        self.qkv_proj  = nn.Linear(d_model, 3 * d_model)
        self.out_proj  = nn.Linear(d_model, d_model)
        self.proj_drop = nn.Dropout(dropout)
        
        # Mark out_proj as a residual projection
        self.out_proj.SCALE_INIT = 1.0
    
    def forward(self, x):
        B, T, C = x.shape
        
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)
        
        def reshape_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        q = reshape_heads(q)
        k = reshape_heads(k)
        v = reshape_heads(v)
        
       
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.proj_drop(self.out_proj(out))
        return out


class FeedForward(nn.Module):
    """2-layer MLP with ReLU."""
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )
        # Mark the second Linear as a residual projection
        self.net[2].SCALE_INIT = 1.0
    
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = FeedForward(d_model, dropout)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.n_layers   = n_layers
        
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop    = nn.Dropout(dropout)
        
        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, n_heads, block_size, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
       
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'SCALE_INIT'):
                std /= (2 * self.n_layers) ** 0.5
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        
        x = self.tok_emb(idx) + self.pos_emb(positions)
        x = self.drop(x)
        x = self.blocks(x)
        x = self.ln_f(x)
        return x


class GPTLanguageModel(nn.Module):
    """
    Autoregressive language model:
        backbone -> linear vocab head
    """
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size, dropout):
        super().__init__()
        self.backbone = MiniGPT(vocab_size, d_model, n_heads, n_layers, block_size, dropout)
        self.lm_head  = nn.Linear(d_model, vocab_size)
        
      
        self.lm_head.weight = self.backbone.tok_emb.weight
    
    def forward(self, idx, targets=None):
        hidden = self.backbone(idx)
        logits = self.lm_head(hidden)
        
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, V),
                targets.reshape(B * T)
            )
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        self.eval()
        for _ in range(max_new_tokens):
            ctx = idx[:, -self.backbone.block_size:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / temperature
            probs  = F.softmax(logits, dim=-1)
            nxt    = torch.multinomial(probs, num_samples=1)
            idx    = torch.cat([idx, nxt], dim=1)
        return idx




def make_loader(dataset, batch_size, shuffle, drop_last):
    """Create a DataLoader."""
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory
    )


def train_epoch(model, loader, optimizer, device, max_grad_norm=1.0):
    """Train for one epoch."""
    model.train()
    total_loss, n_batches = 0.0, 0
    
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        
        total_loss += loss.item()
        n_batches  += 1
    
    return total_loss / max(1, n_batches)


@torch.no_grad()
def eval_loss(model, loader, device):
    """Evaluate loss on a dataset."""
    model.eval()
    total_loss, n_batches = 0.0, 0
    
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        
        _, loss = model(x, y)
        total_loss += loss.item()
        n_batches  += 1
    
    return total_loss / max(1, n_batches)


def plot_curves(train_vals, val_vals, ylabel, title, save_path):
    """Plot and save training curves."""
    epochs = range(1, len(train_vals) + 1)
    
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_vals, "o-", label="Train")
    plt.plot(epochs, val_vals,   "o-", label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"[Plot] Saved -> {save_path}")




def main():
    device = cfg.device
    print(f"Device: {device}\n")
    
    print("=" * 70)
    print(" PART B: Language Model Training")
    print("=" * 70)
    
    # Step 1: Load preprocessed data from Part A
    print("\n[Step 1] Loading preprocessed data...")
    
    with open(cfg.train_tokens_file, "rb") as f:
        train_ids = pickle.load(f)
    with open(cfg.val_tokens_file, "rb") as f:
        val_ids = pickle.load(f)
    
    train_ids = torch.tensor(train_ids, dtype=torch.long)
    val_ids   = torch.tensor(val_ids, dtype=torch.long)
    
    print(f"[Data]  Train tokens: {len(train_ids):,}")
    print(f"[Data]  Val tokens  : {len(val_ids):,}")
    
    # Step 2: Create datasets and dataloaders
    print("\n[Step 2] Creating datasets...")
    
    train_lm_ds = LMDataset(train_ids, cfg.block_size, stride=cfg.lm_stride)
    val_lm_ds   = LMDataset(val_ids,   cfg.block_size, stride=cfg.lm_stride)
    
    train_lm_loader = make_loader(
        train_lm_ds, cfg.lm_batch_size, shuffle=True, drop_last=True
    )
    val_lm_loader = make_loader(
        val_lm_ds, cfg.lm_batch_size, shuffle=False, drop_last=False
    )
    
    print(f"[Dataset]  Train samples: {len(train_lm_ds):,}")
    print(f"[Dataset]  Val samples  : {len(val_lm_ds):,}")
    print(f"[Loader]   Train batches/epoch: {len(train_lm_loader):,}")
    print(f"[Loader]   Val batches/epoch  : {len(val_lm_loader):,}")
    
    # Step 3: Create model
    print("\n[Step 3] Creating model...")
    
    lm_model = GPTLanguageModel(
        vocab_size=cfg.vocab_size,
        d_model=cfg.d_model,
        n_heads=cfg.n_heads,
        n_layers=cfg.n_layers,
        block_size=cfg.block_size,
        dropout=cfg.dropout,
    ).to(device)
    
    n_params = sum(p.numel() for p in lm_model.parameters())
    print(f"[Model]  Parameters: {n_params:,}")
    
    # Step 4: Train
    print("\n[Step 4] Training...")
    
    optimizer = torch.optim.AdamW(lm_model.parameters(), lr=cfg.lm_lr)
    
    lm_train_losses, lm_val_losses = [], []
    
    for epoch in range(1, cfg.lm_epochs + 1):
        tr_loss = train_epoch(lm_model, train_lm_loader, optimizer, device)
        va_loss = eval_loss(lm_model, val_lm_loader, device)
        ppl = math.exp(va_loss)
        
        lm_train_losses.append(tr_loss)
        lm_val_losses.append(va_loss)
        
        print(
            f"  Epoch {epoch:>2}/{cfg.lm_epochs}  |  "
            f"Train Loss {tr_loss:.4f}  |  "
            f"Val Loss {va_loss:.4f}  |  "
            f"Perplexity {ppl:.2f}"
        )
    
    # Step 5: Save model
    print("\n[Step 5] Saving model...")
    torch.save(lm_model.state_dict(), cfg.model_save_path)
    print(f"[Model]  Weights saved -> {cfg.model_save_path}")
    
    # Step 6: Final evaluation
    print("\n[Step 6] Final evaluation...")
    final_val_loss = eval_loss(lm_model, val_lm_loader, device)
    final_ppl = math.exp(final_val_loss)
    print(f"[Model]  Final Validation Perplexity: {final_ppl:.2f}")
    
    # Step 7: Plot training curves
    print("\n[Step 7] Plotting training curves...")
    plot_curves(
        lm_train_losses, lm_val_losses,
        ylabel="Loss",
        title="Language Model — Training Curve",
        save_path=cfg.loss_plot_path
    )
    
    # Step 8: Generate text
    print("\n[Step 8] Generating text...")
    
    sp = spm.SentencePieceProcessor()
    sp.Load(f"{cfg.tokenizer_prefix}.model")
    
    seed_text = "भारत"
    seed_ids = sp.EncodeAsIds(seed_text)
    ctx = torch.tensor([seed_ids], dtype=torch.long, device=device)
    out_ids = lm_model.generate(ctx, max_new_tokens=50, temperature=0.8)
    
    generated_only_ids = out_ids[0].tolist()[len(seed_ids):]
    gen_text = sp.DecodeIds(generated_only_ids)
    
    print(f"\n--- Generated Continuation (50 new tokens) ---\n{gen_text}\n")
    
    with open(cfg.generated_text_path, "w", encoding="utf-8") as f:
        f.write(gen_text)
    print(f"[Generation]  Saved -> {cfg.generated_text_path}")
    
    # Summary
    print("\n" + "=" * 70)
    print(" TRAINING COMPLETE")
    print("=" * 70)
    print(f"  Final Perplexity: {final_ppl:.2f}")
    print("\n  Generated files:")
    print(f"    • {cfg.model_save_path}")
    print(f"    • {cfg.loss_plot_path}")
    print(f"    • {cfg.generated_text_path}")
    print("\n  Use the model weights in Part C (Classification)")


if __name__ == "__main__":
    main()

In [ ]:
"""
=============================================================================
Part C: Text Classification with Pretrained GPT Backbone
CSE556 NLP, Winter 2026 — Assignment 2
=============================================================================
This script:
1. Loads the pretrained GPT backbone from Part B
2. Adds a classification head
3. Fine-tunes (or freezes backbone) for sentiment classification
4. Evaluates and saves results

Note: Uses LAST TOKEN hidden state for classification (assignment requirement)
=============================================================================
"""

import os
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
import sentencepiece as spm



class ConfigC:
    """Configuration for classification training."""
    
    
    tokenizer_prefix     = "/kaggle/input/datasets/madhavgaur12321/data-data/hindi_bpe_trainsplit"
    cls_data_file        = "/kaggle/input/datasets/madhavgaur12321/pre-req/classification_data.pkl"
    pretrained_lm_path   = "/kaggle/input/datasets/madhavgaur12321/pre-req/lm_model.pt"
    
  
    block_size = 128
    d_model    = 256
    n_heads    = 4
    n_layers   = 10
    dropout    = 0.1
    vocab_size = 5000
    num_classes = 3  
    
    
    cls_batch_size = 32
    cls_epochs     = 30       
    cls_lr         = 5e-5     
    
  
    unfreeze_last_n_layers = 2
    
    
    cls_dropout     = 0.5     
    weight_decay    = 0.05    
    label_smoothing = 0.1     
    
   
    patience = 7
    
   
    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    
    num_workers = 0
    pin_memory  = False
    
  
    model_save_path = "cls_model.pt"
    loss_plot_path  = "cls_loss.png"
    predictions_path = "correct_predictions.txt"


cfg = ConfigC()


torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)



class CausalSelfAttention(nn.Module):
    """Multi-head self-attention with a causal mask."""
    def __init__(self, d_model: int, n_heads: int, block_size: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        
        self.qkv_proj  = nn.Linear(d_model, 3 * d_model)
        self.out_proj  = nn.Linear(d_model, d_model)
        self.proj_drop = nn.Dropout(dropout)
        
        self.out_proj.SCALE_INIT = 1.0
    
    def forward(self, x):
        B, T, C = x.shape
        
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)
        
        def reshape_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        q = reshape_heads(q)
        k = reshape_heads(k)
        v = reshape_heads(v)
        
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.proj_drop(self.out_proj(out))
        return out


class FeedForward(nn.Module):
    """2-layer MLP with ReLU."""
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )
        self.net[2].SCALE_INIT = 1.0
    
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-norm transformer block."""
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = FeedForward(d_model, dropout)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    """GPT backbone."""
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.n_layers   = n_layers
        
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop    = nn.Dropout(dropout)
        
        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, n_heads, block_size, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'SCALE_INIT'):
                std /= (2 * self.n_layers) ** 0.5
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        
        x = self.tok_emb(idx) + self.pos_emb(positions)
        x = self.drop(x)
        x = self.blocks(x)
        x = self.ln_f(x)
        return x


class GPTLanguageModel(nn.Module):
    """Language model (needed to load pretrained weights)."""
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size, dropout):
        super().__init__()
        self.backbone = MiniGPT(vocab_size, d_model, n_heads, n_layers, block_size, dropout)
        self.lm_head  = nn.Linear(d_model, vocab_size)
        self.lm_head.weight = self.backbone.tok_emb.weight
    
    def forward(self, idx, targets=None):
        hidden = self.backbone(idx)
        logits = self.lm_head(hidden)
        
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, V),
                targets.reshape(B * T)
            )
        return logits, loss




class GPTClassifier(nn.Module):
    """
    Classifier on top of the GPT backbone.
    
    Uses the LAST TOKEN hidden state for classification.
    This is appropriate for decoder-only models with LEFT-padding,
    where the last position always contains the final real token.
    """
    def __init__(self, backbone: MiniGPT, d_model: int, num_classes: int, 
                 dropout: float = 0.3, label_smoothing: float = 0.0):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.cls_head = nn.Linear(d_model, num_classes)
        self.label_smoothing = label_smoothing
    
    def forward(self, idx, targets=None):
        
        backbone_frozen = not any(p.requires_grad for p in self.backbone.parameters())
        
    
        if backbone_frozen:
            with torch.no_grad():
                hidden = self.backbone(idx)
        else:
            hidden = self.backbone(idx)
       
        
      
        last_hidden = hidden[:, -1, :]
       
        
       
        last_hidden = self.dropout(last_hidden)
        logits = self.cls_head(last_hidden)
        
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits, 
                targets,
                label_smoothing=self.label_smoothing
            )
        return logits, loss



class ClassificationDataset(Dataset):
    """
    Produces (token_ids, label) pairs for sentiment classification.
    
    Uses LEFT-padding so the LAST position always contains the final real token.
    This is critical for last-token classification in decoder-only models.
    """
    def __init__(self, texts, labels, sp, block_size):
        self.texts      = texts
        self.labels     = labels
        self.sp         = sp
        self.block_size = block_size
        self.pad_id     = sp.pad_id()
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        ids = self.sp.EncodeAsIds(self.texts[idx])
        
       
        ids = ids[: self.block_size]
        
    
        pad_len = self.block_size - len(ids)
        ids = [self.pad_id] * pad_len + ids
        
        x = torch.tensor(ids, dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y




def make_loader(dataset, batch_size, shuffle, drop_last):
    """Create a DataLoader."""
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory
    )


def train_epoch(model, loader, optimizer, device, max_grad_norm=1.0):
    """Train for one epoch."""
    model.train()
    total_loss, n_batches = 0.0, 0
    
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        
        total_loss += loss.item()
        n_batches  += 1
    
    return total_loss / max(1, n_batches)


@torch.no_grad()
def eval_loss(model, loader, device):
    """Evaluate loss."""
    model.eval()
    total_loss, n_batches = 0.0, 0
    
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        
        _, loss = model(x, y)
        total_loss += loss.item()
        n_batches  += 1
    
    return total_loss / max(1, n_batches)


@torch.no_grad()
def eval_accuracy(model, loader, device):
    """Evaluate accuracy."""
    model.eval()
    correct, total = 0, 0
    
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        
        logits, _ = model(x)
        preds = logits.argmax(dim=-1)
        
        correct += (preds == y).sum().item()
        total   += y.size(0)
    
    return correct / max(1, total)


def train_with_early_stopping(model, train_loader, val_loader, optimizer, 
                               scheduler, device, epochs, patience=5):
    """
    Training loop with early stopping.
    """
    best_val_acc = 0.0
    patience_counter = 0
    best_model_state = None
    
    train_losses, val_losses = [], []
    
    for epoch in range(1, epochs + 1):
        # Train
        tr_loss = train_epoch(model, train_loader, optimizer, device)
        
        # Evaluate
        va_loss = eval_loss(model, val_loader, device)
        acc = eval_accuracy(model, val_loader, device)
        
        # Update scheduler
        if scheduler is not None:
            scheduler.step()
            current_lr = scheduler.get_last_lr()[0]
        else:
            current_lr = optimizer.param_groups[0]['lr']
        
        train_losses.append(tr_loss)
        val_losses.append(va_loss)
        
        print(
            f"  Epoch {epoch:>2}/{epochs}  |  "
            f"Train Loss {tr_loss:.4f}  |  "
            f"Val Loss {va_loss:.4f}  |  "
            f"Accuracy {acc:.4f}  |  "
            f"LR {current_lr:.2e}"
        )
        
        # Early stopping check
        if acc > best_val_acc:
            best_val_acc = acc
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"    → New best accuracy: {best_val_acc:.4f} ✓")
        else:
            patience_counter += 1
            print(f"    → No improvement ({patience_counter}/{patience})")
            
            if patience_counter >= patience:
                print(f"\n  ⚠️ Early stopping triggered at epoch {epoch}")
                print(f"  Best validation accuracy: {best_val_acc:.4f}")
                
                if best_model_state is not None:
                    model.load_state_dict(best_model_state)
                    print("  Restored best model checkpoint")
                break
    
    return train_losses, val_losses, best_val_acc


def plot_curves(train_vals, val_vals, ylabel, title, save_path):
    """Plot training curves."""
    epochs = range(1, len(train_vals) + 1)
    
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_vals, "o-", label="Train")
    plt.plot(epochs, val_vals,   "o-", label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"[Plot] Saved -> {save_path}")


def freeze_backbone_layers(cls_model, unfreeze_last_n: int):
    """
    Freeze backbone layers except the last N transformer blocks.
    """
    backbone = cls_model.backbone
    n_blocks = len(backbone.blocks)
    
    if unfreeze_last_n == 0:
        print("[Freezing] Backbone FROZEN — training head only")
        for param in backbone.parameters():
            param.requires_grad = False
    
    elif unfreeze_last_n >= n_blocks:
        print(f"[Freezing] Backbone UNFROZEN — fine-tuning all {n_blocks} layers")
        for param in backbone.parameters():
            param.requires_grad = True
    
    else:
        print(f"[Freezing] Unfreezing last {unfreeze_last_n}/{n_blocks} layers")
        
        # Freeze embeddings
        for param in backbone.tok_emb.parameters():
            param.requires_grad = False
        for param in backbone.pos_emb.parameters():
            param.requires_grad = False
        print("  ✗ Embeddings frozen")
        
        # Freeze all transformer blocks initially
        for block in backbone.blocks:
            for param in block.parameters():
                param.requires_grad = False
        
        # Unfreeze ONLY the last N blocks
        for i in range(n_blocks - unfreeze_last_n, n_blocks):
            for param in backbone.blocks[i].parameters():
                param.requires_grad = True
            print(f"  ✓ Block {i+1}/{n_blocks} unfrozen")
        
        # Unfreeze final layer norm
        for param in backbone.ln_f.parameters():
            param.requires_grad = True
        print("  ✓ Final LayerNorm unfrozen")
    
    # Classification head is always trainable
    for param in cls_model.cls_head.parameters():
        param.requires_grad = True
    for param in cls_model.dropout.parameters():
        param.requires_grad = True
    print("  ✓ Classification head trainable")




def main():
    device = cfg.device
    print(f"Device: {device}\n")
    
    print("=" * 70)
    print(" PART C: Text Classification (Last Token)")
    print("=" * 70)
    
   
    print("\n[Step 1] Loading tokenizer...")
    sp = spm.SentencePieceProcessor()
    sp.Load(f"{cfg.tokenizer_prefix}.model")
    pad_id = sp.pad_id()
    print(f"[Tokenizer]  Vocab size = {sp.GetPieceSize()}")
    print(f"[Tokenizer]  Pad ID = {pad_id}")
    
   
    print("\n[Step 2] Loading classification data...")
    with open(cfg.cls_data_file, "rb") as f:
        cls_data = pickle.load(f)
    
    train_cls_ds = ClassificationDataset(
        cls_data['train']['texts'],
        cls_data['train']['labels'],
        sp,
        cfg.block_size
    )
    val_cls_ds = ClassificationDataset(
        cls_data['val']['texts'],
        cls_data['val']['labels'],
        sp,
        cfg.block_size
    )
    
    train_cls_loader = make_loader(
        train_cls_ds, cfg.cls_batch_size, shuffle=True, drop_last=True
    )
    val_cls_loader = make_loader(
        val_cls_ds, cfg.cls_batch_size, shuffle=False, drop_last=False
    )
    
    print(f"[Dataset]  Train: {len(train_cls_ds)}  |  Val: {len(val_cls_ds)}")
    

    train_labels = cls_data['train']['labels']
    val_labels = cls_data['val']['labels']
    print(f"[Dataset]  Train labels: { {i: train_labels.count(i) for i in range(3)} }")
    print(f"[Dataset]  Val labels:   { {i: val_labels.count(i) for i in range(3)} }")
    
   
    print("\n[Step 3] Loading pretrained language model...")
    
    lm_model = GPTLanguageModel(
        vocab_size=cfg.vocab_size,
        d_model=cfg.d_model,
        n_heads=cfg.n_heads,
        n_layers=cfg.n_layers,
        block_size=cfg.block_size,
        dropout=cfg.dropout,
    )
    
    lm_model.load_state_dict(torch.load(cfg.pretrained_lm_path, map_location=device))
    print(f"[Model]  Loaded pretrained weights from {cfg.pretrained_lm_path}")
    
    pretrained_params = sum(p.numel() for p in lm_model.parameters())
    print(f"[Model]  Pretrained model parameters: {pretrained_params:,}")
    
   
    print("\n[Step 4] Creating classifier (LAST TOKEN classification)...")
    
    cls_model = GPTClassifier(
        backbone=lm_model.backbone,
        d_model=cfg.d_model,
        num_classes=cfg.num_classes,
        dropout=cfg.cls_dropout,
        label_smoothing=cfg.label_smoothing
    ).to(device)
    

    freeze_backbone_layers(cls_model, cfg.unfreeze_last_n_layers)
    
    # Count trainable parameters
    n_trainable = sum(p.numel() for p in cls_model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in cls_model.parameters())
    print(f"\n[Model]  Trainable parameters: {n_trainable:,} / {n_total:,} "
          f"({100*n_trainable/n_total:.1f}%)")
    
   
    print("\n[Step 5] Training...")
    print(f"  Classification: Using LAST TOKEN hidden state")
    print(f"  Epochs: {cfg.cls_epochs} (with early stopping, patience={cfg.patience})")
    print(f"  Learning rate: {cfg.cls_lr}")
    print(f"  Weight decay: {cfg.weight_decay}")
    print(f"  Dropout: {cfg.cls_dropout}")
    print(f"  Label smoothing: {cfg.label_smoothing}")
    print()
    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, cls_model.parameters()),
        lr=cfg.cls_lr,
        weight_decay=cfg.weight_decay
    )
    
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.cls_epochs)
    
    cls_train_losses, cls_val_losses, best_acc = train_with_early_stopping(
        model=cls_model,
        train_loader=train_cls_loader,
        val_loader=val_cls_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=cfg.cls_epochs,
        patience=cfg.patience
    )
    
   
    print("\n[Step 6] Saving model...")
    torch.save(cls_model.state_dict(), cfg.model_save_path)
    print(f"[Model]  Weights saved -> {cfg.model_save_path}")
    

    print("\n[Step 7] Final evaluation...")
    final_acc = eval_accuracy(cls_model, val_cls_loader, device)
    final_loss = eval_loss(cls_model, val_cls_loader, device)
    print(f"[Model]  Final Validation Loss: {final_loss:.4f}")
    print(f"[Model]  Final Validation Accuracy: {final_acc:.4f}")
    
  
    print("\n[Step 8] Plotting training curves...")
    plot_curves(
        cls_train_losses, cls_val_losses,
        ylabel="Loss",
        title=f"Classifier (Last Token) — Last {cfg.unfreeze_last_n_layers} layers unfrozen",
        save_path=cfg.loss_plot_path
    )
    
    print("\n[Step 9] Saving sample predictions...")
    
    label_names = {0: "Negative", 1: "Neutral", 2: "Positive"}
    correct_samples = []
    incorrect_samples = []
    cls_model.eval()
    
    va_texts  = cls_data['val']['texts']
    va_labels = cls_data['val']['labels']
    
    with torch.no_grad():
        for i in range(len(va_texts)):
            ids = sp.EncodeAsIds(va_texts[i])[:cfg.block_size]
            pad = cfg.block_size - len(ids)
            ids = [pad_id] * pad + ids
            
            x = torch.tensor([ids], dtype=torch.long, device=device)
            logits, _ = cls_model(x)
            pred = logits.argmax(dim=-1).item()
            
            sample = (
                va_texts[i],
                label_names[va_labels[i]],
                label_names[pred]
            )
            
            if pred == va_labels[i]:
                if len(correct_samples) < 5:
                    correct_samples.append(sample)
            else:
                if len(incorrect_samples) < 5:
                    incorrect_samples.append(sample)
    
    with open(cfg.predictions_path, "w", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write("CORRECT PREDICTIONS\n")
        f.write("=" * 60 + "\n\n")
        
        for j, (rev, true, pred) in enumerate(correct_samples, 1):
            f.write(f"=== Sample {j} ===\n")
            f.write(f"Review   : {rev}\n")
            f.write(f"True     : {true}\n")
            f.write(f"Predicted: {pred} ✓\n\n")
        
        f.write("\n" + "=" * 60 + "\n")
        f.write("INCORRECT PREDICTIONS\n")
        f.write("=" * 60 + "\n\n")
        
        for j, (rev, true, pred) in enumerate(incorrect_samples, 1):
            f.write(f"=== Sample {j} ===\n")
            f.write(f"Review   : {rev}\n")
            f.write(f"True     : {true}\n")
            f.write(f"Predicted: {pred} ✗\n\n")
    
    print(f"[Predictions]  Saved {len(correct_samples)} correct, {len(incorrect_samples)} incorrect")
    print(f"[Predictions]  File -> {cfg.predictions_path}")
    
 
    print("\n" + "=" * 70)
    print(" CLASSIFICATION COMPLETE")
    print("=" * 70)
    print(f"  Classification method: LAST TOKEN hidden state")
    print(f"  Dataset size:          {len(train_cls_ds)} train / {len(val_cls_ds)} val")
    print(f"  Trainable params:      {n_trainable:,} / {n_total:,}")
    print(f"  Unfrozen layers:       Last {cfg.unfreeze_last_n_layers} / {cfg.n_layers}")
    print(f"  Best Accuracy:         {best_acc:.4f}")
    print(f"  Final Accuracy:        {final_acc:.4f}")
    print(f"\n  Regularization:")
    print(f"    • Dropout:           {cfg.cls_dropout}")
    print(f"    • Weight decay:      {cfg.weight_decay}")
    print(f"    • Label smoothing:   {cfg.label_smoothing}")
    print(f"    • Early stopping:    patience={cfg.patience}")
    print(f"\n  Generated files:")
    print(f"    • {cfg.model_save_path}")
    print(f"    • {cfg.loss_plot_path}")
    print(f"    • {cfg.predictions_path}")


if __name__ == "__main__":
    main()